# GH Copilot in Action for APIM & AI Foundry — Backend Pool Load Balancing

This notebook walks through the  deployment of Azure API Management (APIM) as an AI Gateway in front of multiple Azure AI Foundry (Azure OpenAI–compatible) endpoints using APIM's **built-in backend pool** feature with **priority-based routing**.

## Overview: How It Works

- APIM exposes a single inference endpoint to clients.

- A backend pool groups **four** AI Foundry deployments across two different regions at **three priority levels**.

- The **Priority 1** backend (PTU DZ in Sweden Central) receives all traffic under normal conditions.
- When it returns **HTTP 429** (rate-limited), APIM's `retry` policy transparently retries against the pool — the two **Priority 2** backends (PayGo DZ in Sweden Central and PayGo DZ in Germany West Central) each receive ~50% of the overflowed traffic, balanced by `weight`.
- If all Priority 2 backends are also exhausted, traffic falls to the **Priority 3** backend (PayGo Global in Germany West Central) as a last resort.
- Only if **all backends are unavailable** does APIM return **HTTP 503** to the caller.
- **Objective is to reflect no 429 errors to the clients.**

## Recommended Architecture

```
Caller (Python SDK / HTTP client)
        Caller (Python SDK / HTTP client)
                │
                ▼
          Azure API Management (Basicv2 tier)
          ┌──────────────────────────────────────────────────────────┐
          │  Inference API  /inference/openai/...                    │
          │  Policy:                                                 │
          │    • set-backend-service → backend pool                  │
          │    • retry on 429 / 503 (count=2, tries all backends)    │
          └───────┬──────────────────────────────────────────────────┘
                  │
                  ▼
           ┌─────────────────────────────────────────────┐
           │  Backend Pool (priority + weighted routing) │
           │                                             │
           │  ┌─────────────────────────────────────┐   │
           │  │ Priority 1                          │   │  ← served first
           │  │ PTU DZ - swedencentral              │   │
           │  └─────────────────────────────────────┘   │
           │                                             │
           │  ┌──────────────────┐ ┌──────────────────┐ │
           │  │ Priority 2 w=50  │ │ Priority 2 w=50  │ │  ← 50/50 on P1 failover
           │  │ PayGo DZ         │ │ PayGo DZ         │ │
           │  │ swedencentral    │ │ germanywestcent  │ │
           │  └──────────────────┘ └──────────────────┘ │
           │                                             │
           │  ┌─────────────────────────────────────┐   │
           │  │ Priority 3                          │   │  ← last resort
           │  │ PayGo Global Germany West           │   │
           │  └─────────────────────────────────────┘   │
           └─────────────────────────────────────────────┘
```

> ⚠️ **WARNING:** The demo architecture will include only **PayGo** deployments of **gpt-4o-mini** due to limitations in the Sandbox environment.

## Demo Architecture

```
Caller (Python SDK / HTTP client)
        Caller (Python SDK / HTTP client)
                │
                ▼
          Azure API Management (Basicv2 tier)
          ┌──────────────────────────────────────────────────────────┐
          │  Inference API  /inference/openai/...                    │
          │  Policy:                                                 │
          │    • set-backend-service → backend pool                  │
          │    • retry on 429 / 503 (count=2, tries all backends)    │
          └───────┬──────────────────────────────────────────────────┘
                  │
                  ▼
           ┌─────────────────────────────────────────────┐
           │  Backend Pool (priority + weighted routing) │
           │                                             │
           │  ┌─────────────────────────────────────┐   │
           │  │ Priority 1                          │   │  ← served first
           │  │ PTU DZ - swedencentral              │   │
           │  └─────────────────────────────────────┘   │
           │                                             │
           │  ┌──────────────────┐ ┌──────────────────┐ │
           │  │ Priority 2 w=50  │ │ Priority 2 w=50  │ │  ← 50/50 on P1 failover
           │  │ PayGo DZ         │ │ PayGo DZ         │ │
           │  │ swedencentral    │ │ germanywestcent  │ │
           │  └──────────────────┘ └──────────────────┘ │
           │                                             │
           │  ┌─────────────────────────────────────┐   │
           │  │ Priority 3                          │   │  ← last resort
           │  │ PayGo Global Germany West           │   │
           │  └─────────────────────────────────────┘   │
           └─────────────────────────────────────────────┘
```

## Step 0: Prerequisites

Complete this section before running any code cells.

### Required Tools

| Tool | Version | Install |
|------|---------|---------|
| Python | 3.12+ | [python.org](https://www.python.org/downloads/) |
| VS Code | Latest | [code.visualstudio.com](https://code.visualstudio.com/) |
| VS Code Jupyter extension | Latest | [Marketplace](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) |
| Azure CLI | Latest | `curl -sL https://aka.ms/InstallAzureCLIDeb \| sudo bash` |
| Git | Any | `sudo apt install git` |

> **Azure CLI must be signed in** before deployment steps. Run `az login` first.
> The first setup code cell in **Step 1** creates `.venv`, installs dependencies, and registers the notebook kernel.

### Azure Permissions

Your account needs **both** on the target subscription or resource group:
- `Contributor` — to create resources
- `RBAC Administrator` (or `Owner`) — to assign RBAC roles to the APIM managed identity

> **Why?** The APIM system-assigned managed identity must be granted the **Cognitive Services OpenAI User** role on each AI Foundry account so it can obtain Entra ID tokens for authentication.

### Azure Regions with Model Availability

All regions must support the model you intend to deploy. Verify availability at:
[Azure OpenAI models by region](https://learn.microsoft.com/azure/ai-services/openai/concepts/models)

Regions used in this lab: `swedencentral`, `germanywestcentral`.

---

## Step 0a: Verify Prerequisites (CLI Checklist)

Run the commands below in a terminal to verify all prerequisites are installed. Install any missing tools before proceeding.

### ✓ Python 3.12+

```bash
python --version
# Expected: Python 3.12.x or higher
```

### ✓ Azure CLI

```bash
az --version
# Expected: azure-cli version 2.60.0 or higher
```

If not installed:
```bash
pip install azure-cli
```

Then sign in:
```bash
az login
```

### ✓ Git

```bash
git --version
# Expected: git version 2.x or higher
```

### ✓ VS Code & Jupyter Extension

```bash
code --version
# Expected: version 1.x or higher
```

Install the Jupyter extension:
- Open VS Code
- Press `Ctrl+Shift+X` (Windows/Linux) or `Cmd+Shift+X` (macOS)
- Search for "Jupyter"
- Click **Install**

### ✓ Azure Permissions

Verify you're logged into the right subscription:

```bash
az account show --query "{User:user.name, Subscription:name}" -o table
```

Your account needs **Contributor** and **RBAC Administrator** roles on the target subscription.

---

Ready? Proceed to **Step 1**.

---

## Step 1: Create venv and Install Local Dependencies

> **First-time kernel selection:** When you run the cell below, VS Code will ask you to choose a kernel. Pick **Python 3** (the default system interpreter). This is only used to bootstrap the virtual environment — you will switch to the `.venv` kernel in Step 2.

This repository is self-contained for this lab. No external repository clone is required.

Expected layout after this step:
```
/workspaces/
└── from-specs-to-mission-critical-deployments/
    ├── main.bicep
    ├── modules/                 ← local Bicep modules
    ├── shared/                  ← local Python helpers
    ├── policy.xml
    └── .venv/                   ← created by this step
```

This step will:
1. Create `.venv` from the system Python
2. Install `requirements.txt` into `.venv`

Run the setup code below:

In [2]:
import subprocess
import os
import sys

# From your demo repo root
repo_root = os.getcwd()
print(f"Working from: {repo_root}\n")

# --- 1. Create virtual environment ---
print("1. Creating virtual environment...")
venv_path = os.path.join(repo_root, ".venv")
result = subprocess.run(
    [sys.executable, "-m", "venv", venv_path],
    capture_output=True,
    text=True
)
if result.returncode != 0:
    print(f"❌ Failed to create .venv:\n{result.stderr}")
    raise SystemExit("Cannot continue without .venv")

venv_python = os.path.join(venv_path, "bin", "python")
if not os.path.isfile(venv_python):
    print(f"❌ Expected venv python at {venv_python} but it does not exist.")
    raise SystemExit("Cannot continue without .venv/bin/python")
print(f"✓ Virtual environment ready at {venv_path}")

# --- 2. Install dependencies ---
print("\n2. Installing dependencies from requirements.txt...")
result = subprocess.run(
    [venv_python, "-m", "pip", "install", "-q", "--upgrade", "pip"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"⚠️ pip upgrade warning: {result.stderr}")

result = subprocess.run(
    [venv_python, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=repo_root,
    capture_output=True,
    text=True
)
if result.returncode != 0:
    print(f"❌ pip install failed:\n{result.stdout}\n{result.stderr}")
    raise SystemExit("Dependency installation failed")
print("✓ Dependencies installed")

# --- Next steps ---
print("\n" + "=" * 60)
print("✅ Setup complete!")
print("   Click the kernel picker (top-right) and select:")
print("     .venv (Python 3.12.x)")
print("   Then run the next cell (Step 2) to verify.")
print("=" * 60)

Working from: /workspaces/from-specs-to-mission-critical-deployments

1. Creating virtual environment...
✓ Virtual environment ready at /workspaces/from-specs-to-mission-critical-deployments/.venv

2. Installing dependencies from requirements.txt...
✓ Dependencies installed

✅ Setup complete!  Follow these steps to continue:

  1. Reload VS Code:  Ctrl+Shift+P → 'Developer: Reload Window'
  2. After reload, click the kernel picker (top-right of this notebook)
  3. Select:  .venv (Python 3.12.x)
  4. Run the next cell (Step 2 verification) to confirm it worked


---

## Step 2: Select the `.venv` Kernel and Verify

After running Step 1, switch the notebook kernel to the `.venv` environment:

1. **Click the kernel picker** (top-right corner of the notebook)
2. **Select:** `.venv (Python 3.12.x)`

Then run the verification cell below to confirm it is active and all dependencies are installed.

In [1]:
import sys, os

# Verify the kernel points to the .venv environment
venv_path = os.path.join(os.getcwd(), ".venv")
using_venv = sys.executable.startswith(venv_path)

print(f"Python executable: {sys.executable}")
print(f"Python version:    {sys.version.split()[0]}")
print(f"Using .venv:       {'✅ Yes' if using_venv else '❌ No — select the .venv kernel and re-run this cell'}")

if not using_venv:
    print("\n⚠️  You are NOT running inside the .venv environment.")
    print("    Click the kernel picker (top-right) and choose:")
    print("      • .venv (Python 3.12.x)")
    print("    Then re-run this cell.")
else:
    # Quick import check for required packages
    missing = []
    for pkg in ["requests", "pandas", "matplotlib", "openai"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"\n❌ Missing packages: {', '.join(missing)}")
        print("   Re-run the Step 1 setup cell to install them.")
    else:
        print("\n✅ Kernel is ready. Proceed to Step 3.")

Python executable: /workspaces/from-specs-to-mission-critical-deployments/.venv/bin/python
Python version:    3.12.1
Using .venv:       ✅ Yes

✅ Kernel is ready. Proceed to Step 3.


---

## Step 3: Initialize Notebook Variables

Set deployment names, region configuration, model parameters, and APIM settings.

> **Requires the `.venv` kernel.** If you skipped Step 2, select the `.venv` kernel first.

In [2]:
import os, sys, json

# Add shared utilities from this repository
sys.path.insert(0, os.path.join(os.getcwd(), 'shared'))
import utils

# Deployment naming
deployment_name = "from-specs-to-mission-critical-deployments"
resource_group_name = f"rg-{deployment_name}"
resource_group_location = "swedencentral"

# AI Foundry configuration: 4 accounts across 4 priority levels
aiservices_config = [
    {"name": "Endpoint1-PTU-DZ-Sweden ", "location": "swedencentral", "priority": 1},
    {"name": "Endpoint2-PayGo-DZ-Sweden ", "location": "swedencentral", "priority": 2, "weight": 50},
    {"name": "Endpoint3-PayGo-DZ-Germany ", "location": "germanywestcentral", "priority": 2, "weight": 50},
    {"name": "Endpoint4-PayGo-Global-Germany ", "location": "germanywestcentral", "priority": 3}
 ]

# Model deployment configuration
models_config = [
    {
        "name": "gpt-4o-mini",
        "publisher": "OpenAI",
        "version": "2024-07-18",
        "sku": "GlobalStandard",
        "capacity": 1  # ~1k TPM (intentionally low to trigger failover quickly)
    }
]

# APIM configuration
apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# Inference API configuration
inference_api_path = "inference"
inference_api_type = "AzureOpenAI"  # options: AzureOpenAI, AzureAI, OpenAI, PassThrough
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

utils.print_ok('Notebook initialized')

# Display configuration summary
print("\n📋 Configuration Summary:")
print(f"  Deployment: {deployment_name}")
print(f"  Resource Group: {resource_group_name}")
print(f"  Locations: {[a['location'] for a in aiservices_config]}")
print(f"  Model: {models_config[0]['name']}")
print(f"  APIM SKU: {apim_sku}")

✅ Notebook initialized ⌚ 21:00:07.788158 

📋 Configuration Summary:
  Deployment: from-specs-to-mission-critical-deployments
  Resource Group: rg-from-specs-to-mission-critical-deployments
  Locations: ['swedencentral', 'swedencentral', 'germanywestcentral', 'germanywestcentral']
  Model: gpt-4o-mini
  APIM SKU: Basicv2


---

## Step 4: Verify Azure CLI and Subscription

Before deploying, verify you are logged in to the correct Azure subscription.

> Run `az login` in a terminal if you haven't already. The session persists across restarts.

In [3]:
output = utils.run(
    "az account show",
    "Retrieved az account",
    "Failed to get the current az account"
)

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")
    
    # Store for later use
    tenant_id_str = tenant_id
    subscription_id_str = subscription_id
else:
    print("❌ Failed to retrieve account info. Run 'az login' first.")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 21:00:26.225911 :1s]
👉🏽 Current user: burak-admin@MngEnvMCAP371870.onmicrosoft.com
👉🏽 Tenant ID: 6ee29205-81b5-4e4b-b235-5bd9d6fb6b04
👉🏽 Subscription ID: 9103cd46-543d-4b44-a957-f011acb997c6


---

## Step 5: Deploy Infrastructure with Bicep

This cell creates the resource group and deploys all Azure resources via Bicep template. The deployment includes:

1. **APIM instance** (Basicv2 tier) with system-assigned managed identity
2. **Four AI Foundry accounts** across four regions, each with `gpt-4o-mini` model deployment
3. **RBAC role assignments** — APIM managed identity gets Cognitive Services OpenAI User role
4. **Inference API** with backend pool, retry policy, and circuit breaker

> **Skip this cell if your resources are already deployed.** Jump to Step 6 to retrieve existing deployment outputs.

⏱️ **Expected duration:** 5–10 minutes

In [ ]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": {"value": apim_sku},
        "aiServicesConfig": {"value": aiservices_config},
        "modelsConfig": {"value": models_config},
        "apimSubscriptionsConfig": {"value": apim_subscriptions_config},
        "inferenceAPIPath": {"value": inference_api_path},
        "inferenceAPIType": {"value": inference_api_type},
        "foundryProjectName": {"value": foundry_project_name}
    }
}

# Write the parameters to params.json
with open('params.json', 'w') as f:
    json.dump(bicep_parameters, f, indent=2)

# Run the deployment
print("🚀 Deploying infrastructure...")
output = utils.run(
    f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"✅ Deployment '{deployment_name}' succeeded",
    f"❌ Deployment '{deployment_name}' failed"
)

if output.success:
    print("\n✨ Infrastructure deployed successfully!")
    print(f"   Resource Group: {resource_group_name}")
    print(f"   Deployment: {deployment_name}")
else:
    print("\n⚠️ Deployment failed. Check the errors above and retry.")

---

## Step 6: Retrieve Deployment Outputs

Get the APIM gateway URL, subscription keys, and other deployment outputs needed for testing.

In [ ]:
# Retrieve all outputs from the deployment
output = utils.run(
    f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}"
)

if output.success and output.json_data:
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    
    print("\n📋 Deployment Outputs:")
    print(f"  Service ID: {apim_service_id}")
    print(f"  Gateway URL: {apim_resource_gateway_url}")
    
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        print(f"  Subscription: {subscription_name} (key: ****{subscription_key[-4:]})")
    
    # Store API key for test cells
    api_key = apim_subscriptions[0].get("key")
    inference_api_version_output = inference_api_version
    
    print("\n✅ Outputs retrieved successfully!")
else:
    print("❌ Failed to retrieve outputs.")
    apim_resource_gateway_url = None
    api_key = None

---

## Step 7: Test API with Direct HTTP Requests

Sends 20 sequential requests to the APIM inference endpoint using the `requests` library. Each response includes the `x-ms-region` header showing which backend served the request.

> ⏱️ **Wait 1–2 minutes after deployment** before running this cell to allow model deployments to fully initialize.

This test demonstrates steady-state priority routing under normal (non-bursty) load.

In [ ]:
import requests
import time

runs = 20
sleep_time_ms = 100
url = f"{apim_resource_gateway_url}/{inference_api_path}/openai/deployments/{models_config[0]['name']}/chat/completions?api-version={inference_api_version}"
messages = {"messages": [
    {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
    {"role": "user", "content": "Can you tell me the time, please?"}
]}

api_runs = []

# Initialize session for connection pooling
session = requests.Session()
session.headers.update({'api-key': api_key})

try:
    for i in range(runs):
        print(f"▶️ Run {i+1}/{runs}:")
        
        start_time = time.time()
        response = session.post(url, json=messages)
        response_time = time.time() - start_time
        print(f"⌚ {response_time:.2f} seconds")
        
        utils.print_response_code(response)
        
        if "x-ms-region" in response.headers:
            region = response.headers.get('x-ms-region')
            print(f"x-ms-region: \x1b[1;32m{region}\x1b[0m")
            api_runs.append((response_time, region))
        
        if response.status_code == 200:
            data = json.loads(response.text)
            print(f"Token usage: {json.dumps(dict(data.get('usage')), indent=2)}\n")
            print(f"💬 {data.get('choices')[0].get('message').get('content')}\n")
        else:
            print(f"Error: {response.text}\n")
        
        time.sleep(sleep_time_ms / 1000)
finally:
    session.close()

print(f"\n✅ Sequential test complete! ({len(api_runs)} requests captured)")

---

## Step 8: Analyze Load Balancing Results

Visualizes response times color-coded by region. This shows which backend(s) handled each request. The priority 1 backend (East US) is used until TPM exhaustion, then distribution occurs across priority 2 backends (50/50 split).

> Note: The first request may take longer due to connection warm-up.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle as pltRectangle
import matplotlib as mpl

if api_runs:
    mpl.rcParams['figure.figsize'] = [15, 7]
    df = pd.DataFrame(api_runs, columns=['Response Time', 'Region'])
    df['Run'] = range(1, len(df) + 1)
    
    # Color map for regions
    color_map = {'Sweden Central': 'lightyellow', 'Germany West Central': 'lightcyan'}
    
    # Plot
    ax = df.plot(
        kind='bar',
        x='Run',
        y='Response Time',
        color=[color_map.get(region, 'gray') for region in df['Region']],
        legend=False
    )
    
    # Add legend
    legend_labels = [pltRectangle((0, 0), 1, 1, color=color_map.get(region, 'gray')) for region in df['Region'].unique()]
    ax.legend(legend_labels, df['Region'].unique(), loc='upper left')
    
    plt.title('Load Balancing Results — Sequential Test')
    plt.xlabel('Run #')
    plt.ylabel('Response Time (seconds)')
    plt.xticks(rotation=0)
    
    average = df['Response Time'].mean()
    plt.axhline(y=average, color='r', linestyle='--', label=f'Average: {average:.2f}s')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print region distribution
    print("\n📊 Region Distribution:")
    print(df['Region'].value_counts().to_string())
else:
    print("No test data available. Run Step 7 first.")

---

## Step 9: Test API Using Azure OpenAI Python SDK

Repeats the test using the official Azure OpenAI Python SDK to ensure compatibility. Uses `with_raw_response` to access the `x-ms-region` header.

> ⏱️ **Wait 1–2 minutes** before re-running if you just ran Step 7.

In [ ]:
from openai import AzureOpenAI

runs = 20
sleep_time_ms = 100

client = AzureOpenAI(
    azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
    api_key=api_key,
    api_version=inference_api_version
)

sdk_runs = []

for i in range(runs):
    print(f"▶️ Run {i+1}/{runs}:")
    
    start_time = time.time()
    raw_response = client.chat.completions.with_raw_response.create(
        model=models_config[0]['name'],
        messages=[
            {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
            {"role": "user", "content": "Can you tell me what 2+2 equals?"}
        ]
    )
    response_time = time.time() - start_time
    
    print(f"⌚ {response_time:.2f} seconds")
    
    region = raw_response.headers.get('x-ms-region', 'Unknown')
    print(f"x-ms-region: \x1b[1;32m{region}\x1b[0m")
    sdk_runs.append((response_time, region))
    
    response = raw_response.parse()
    
    if response.usage:
        print(f"Token usage: total={response.usage.total_tokens}, prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}")
    
    print(f"💬 {response.choices[0].message.content}\n")
    
    time.sleep(sleep_time_ms / 1000)

print(f"✅ SDK test complete! ({len(sdk_runs)} requests captured)")

---

## Step 10: Understand the APIM Policy

The APIM **retry policy** transparently intercepts HTTP 429 and 503 responses and reroutes requests to the next available backend in the pool. This ensures callers never see a rate-limit error unless **all** backends are exhausted.

### APIM Policy XML

The policy is applied to the inference API and uses APIM's **backend pool feature** for priority-based routing:

```xml
<policies>
    <inbound>
        <base />
        <set-backend-service backend-id="{backend-id}" />
    </inbound>
    <backend>
        <!--Set count to one less than the number of backends in the pool to try
            all backends until the backend pool is temporarily unavailable.-->
        <retry count="2" interval="0" first-fast-retry="true"
               condition="@(context.Response.StatusCode == 429 ||
                            context.Response.StatusCode == 503)">
            <!--Switch back to same backend pool which will have automatically
                removed the faulty backend -->
            <set-backend-service backend-id="{backend-id}" />
            <forward-request buffer-request-body="true" />
        </retry>
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
        <choose>
            <!--Return a generic error that does not reveal backend pool details.-->
            <when condition="@(context.Response.StatusCode == 503)">
                <return-response>
                    <set-status code="503" reason="Service Unavailable" />
                </return-response>
            </when>
        </choose>
    </on-error>
</policies>
```

### Policy Behavior

| Policy Element | What It Does |
|---|---|
| `set-backend-service` (inbound) | Directs request to the named backend pool instead of individual backends |
| `retry` (backend, count=2) | On HTTP 429 or 503, retries up to 2 times; APIM picks the next available backend by priority |
| `buffer-request-body="true"` | Buffers the request body so it can be replayed to the next backend |
| `on-error choose` | Returns a sanitised 503; caller never sees backend names or URLs |

---

## Step 11: Clean Up Resources

When finished with the lab, remove all deployed resources to avoid Azure charges.

The source lab provides a separate cleanup notebook:
[clean-up-resources.ipynb](https://github.com/Azure-Samples/AI-Gateway/blob/main/labs/backend-pool-load-balancing/clean-up-resources.ipynb)

Alternatively, delete the entire resource group:

```bash
az group delete --name lab-backend-pool-load-balancing --yes
```

This removes:
- AI Foundry projects and accounts
- APIM instance
- All associated networking and compute resources

---

## Troubleshooting

### Deployment stuck / times out

- Basicv2 APIM typically completes in ~2 minutes. If stuck beyond 10 minutes, check the Azure Portal under **Resource group → Deployments** for live status.
- If deployment fails at the APIM step, verify your subscription has capacity for the `Basicv2` SKU in your region.

### Model deployment errors

- Verify `gpt-4o-mini` (version `2024-07-18`) is available in **both regions** at the [Azure OpenAI models page](https://learn.microsoft.com/azure/ai-services/openai/concepts/models).
- `GlobalStandard` SKU requires quota in each region. Check per-region quota:
  ```bash
  for loc in swedencentral germanywestcentral; do
    echo "=== $loc ==="
    az cognitiveservices usage list --location $loc -o table 2>/dev/null | grep -i gpt
  done
  ```

### HTTP 401 on test calls

- The APIM subscription key is passed via the `api-key` header. Confirm the key hasn't been regenerated.
- Double-check `apim_resource_gateway_url` doesn't end with a trailing slash.

### HTTP 503 on test calls

- All four backends are exhausted simultaneously. With `capacity: 1` (~1k TPM each), rapid bursts drain the entire pool.
- Increase `sleep_time_ms` (e.g., to `500`) or wait a minute for quota to reset, then re-run the test cell.
- Increase `capacity` in `models_config` and re-run the deployment cell.

### Kernel failed to start

This happens when the `.venv` was deleted or corrupted. Re-run the Step 1 setup cell to recreate it, then reload VS Code (`Ctrl+Shift+P` → *Developer: Reload Window*).

### x-ms-region header missing

This header is set by each Azure OpenAI backend. If missing, the request may not have reached the backend pool. Check the APIM logs in Azure Portal for details.

---

## Further Reading

- [APIM Backend Pools Documentation](https://learn.microsoft.com/azure/api-management/backends?tabs=bicep)
- [Azure OpenAI Models and Availability](https://learn.microsoft.com/azure/ai-services/openai/concepts/models)
- [APIM Retry Policy Reference](https://learn.microsoft.com/azure/api-management/retry-policy)
- [APIM Managed Identity Authentication](https://learn.microsoft.com/azure/api-management/authentication-managed-identity-policy)
- [AI Gateway Lab Catalogue](https://github.com/Azure-Samples/AI-Gateway/tree/main/labs)
- [Azure AI Foundry Deployment Types](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/concepts/deployment-types)
- [Azure API Management Tier Features](https://learn.microsoft.com/en-us/azure/api-management/api-management-features)

---

## Notes

- This notebook is a faithful conversion of the [source lab guide](https://github.com/Azure-Samples/AI-Gateway/blob/main/labs/backend-pool-load-balancing/backend-pool-load-balancing.ipynb).
- All code cells can run in sequence; cell-to-cell state is preserved via Python variables.
- This repository includes local Bicep modules and shared utilities so no external clone is required for this scenario.
- Questions or issues? See the [main AI-Gateway repo](https://github.com/Azure-Samples/AI-Gateway) or open an issue there.